下面是一份按知识点组织的 DeepSpeed 代码实操教程，涵盖配置参数详解、核心 API、各功能模块的使用方法，帮助你从零开始跑通大模型训练与推理。

---

## 1. 安装与基础环境

```bash
# 通过 pip 安装，自动匹配 CUDA 版本
pip install deepspeed

# 检查安装状态
ds_report
```

---

## 2. 核心 API：deepspeed.initialize 与训练循环

DeepSpeed 的训练入口是 `deepspeed.initialize`，它会根据配置文件自动完成模型、优化器、数据加载器的分布式包装。

```python
import deepspeed
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# 1. 定义模型、优化器、数据加载器
model = MyModel()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
dataloader = DataLoader(dataset, batch_size=8)

# 2. 初始化 DeepSpeed 引擎
model_engine, optimizer, trainloader, _ = deepspeed.initialize(
    model=model,                    # 原始模型
    optimizer=optimizer,            # 原始优化器（也可在配置中定义）
    model_parameters=model.parameters(),  # 用于配置中的参数组（如 ZeRO）
    training_data=dataloader.dataset,    # 可选，为数据加载器注入分布式采样器
    config="ds_config.json"              # 或传入 dict 形式的 config_params
)

# 3. 训练循环
for epoch in range(num_epochs):
    for batch in trainloader:
        loss = model_engine(batch)
        model_engine.backward(loss)      # 替代 loss.backward()
        model_engine.step()              # 替代 optimizer.step()
```

**model_engine 对象** 包装了原始模型，用法完全相同，可被直接调用进行前向计算。

**核心方法**：
- `model_engine.backward(loss)`：自动执行梯度缩放（如果启用 fp16）、通信和反向传播。
- `model_engine.step()`：自动处理梯度累积、All‑Reduce/Reduce‑Scatter、梯度裁剪和参数更新。

---

## 3. 配置文件（ds_config.json）详解

DeepSpeed 通过一个 JSON 文件控制所有优化特性。以下是核心部分及参数说明。

### 3.1 基础训练参数

```json
{
  "train_batch_size": 32,                  // 全局 batch size（所有 GPU 总和）
  "gradient_accumulation_steps": 4,        // 梯度累积步数，模拟大 batch
  "train_micro_batch_size_per_gpu": 4,     // 每张 GPU 每个 step 的 micro-batch size
  "optimizer": {                           // 可在配置中定义优化器（不传 optimizer 时使用）
    "type": "AdamW",
    "params": {
      "lr": 1e-4,
      "betas": [0.9, 0.999],
      "eps": 1e-8,
      "weight_decay": 0.01
    }
  },
  "scheduler": {                           // 学习率调度器
    "type": "WarmupDecayLR",
    "params": {
      "warmup_min_lr": 0,
      "warmup_max_lr": 1e-4,
      "warmup_num_steps": 1000,
      "total_num_steps": 10000
    }
  },
  "fp16": {                                // 混合精度训练
    "enabled": true,
    "loss_scale": 0,                      // 0 表示动态损失缩放
    "initial_scale_power": 16,
    "loss_scale_window": 1000
  },
  "bf16": { "enabled": false }            // BF16 训练（与 fp16 互斥）
}
```

### 3.2 ZeRO 优化配置

```json
{
  "zero_optimization": {
    "stage": 2,                            // 0,1,2,3 （0 禁用 ZeRO）
    "offload_optimizer": {                 // 优化器状态卸载到 CPU
      "device": "cpu",
      "pin_memory": true
    },
    "offload_param": {                     // 参数卸载（通常 stage=3 时使用）
      "device": "cpu",                     // 或 "nvme"
      "pin_memory": true
    },
    "allgather_partitions": true,          // 使用 All-Gather 收集分片参数
    "allgather_bucket_size": 5e8,          // All-Gather 的桶大小（字节）
    "reduce_scatter": true,                // 使用 Reduce-Scatter 聚合梯度
    "reduce_bucket_size": 5e8,             // Reduce 桶大小
    "overlap_comm": true,                  // 通信与计算重叠
    "contiguous_gradients": true,          // 梯度存为连续内存，加速通信
    "round_robin_gradients": true,         // 轮询分片梯度
    "stage3_max_live_parameters": 1e9,     // Stage 3 最多同时活着的参数数量
    "stage3_max_reuse_distance": 1e9,      // Stage 3 参数重用距离
    "stage3_prefetch_bucket_size": 5e8,    // Stage 3 预取桶大小
    "stage3_param_persistence_threshold": 1e6, // 小于此阈值的参数常驻 GPU
    "stage3_gather_16bit_weights_on_model_save": true // 保存时收集 16bit 权重
  }
}
```

**关键参数解释**：
- `stage`：ZeRO 阶段。
  - 1：仅分片优化器状态。
  - 2：分片优化器状态 + 梯度。
  - 3：分片参数 + 梯度 + 优化器状态。
- `offload_optimizer`/`offload_param`：`device` 可选 `"cpu"` 或 `"nvme"`。
  - `"nvme"` 需额外指定 `nvme_path` 和 `aio` 配置。
- `allgather_bucket_size` 和 `reduce_bucket_size`：控制通信的粒度和重叠。
- `overlap_comm`：开启后，通信可以与反向传播计算重叠（需要梯度累积或 Stage 3 预取）。

### 3.3 ZeRO-Infinity 的 NVMe 卸载配置

```json
{
  "zero_optimization": {
    "stage": 3,
    "offload_optimizer": {
      "device": "nvme",
      "nvme_path": "/local/nvme",
      "pin_memory": true,
      "buffer_count": 4,
      "fast_init": false
    },
    "offload_param": {
      "device": "nvme",
      "nvme_path": "/local/nvme",
      "pin_memory": true,
      "buffer_count": 5,
      "buffer_size": 1e8
    },
    "aio": {
      "block_size": 1048576,
      "queue_depth": 8,
      "single_submit": false,
      "overlap_events": true
    }
  }
}
```

**参数说明**：
- `nvme_path`：NVMe 设备挂载点。
- `buffer_count`、`buffer_size`：CPU 端用于预取的缓冲区数量和大小。
- `aio`：异步 I/O 参数。
  - `block_size`：每次 IO 的块大小。
  - `queue_depth`：异步 IO 队列深度。
  - `single_submit`：批量提交 IO 请求。
  - `overlap_events`：允许事件重叠，提高并发。

---

## 4. 通信优化：1-bit Adam

在配置中启用 1-bit Adam，大幅降低跨机通信量。

```json
{
  "optimizer": {
    "type": "OneBitAdam",
    "params": {
      "lr": 1e-4,
      "betas": [0.9, 0.999],
      "eps": 1e-8,
      "weight_decay": 0.01,
      "freeze_step": 1000,          // 前 freeze_step 步使用普通 Adam 热身
      "cuda_aware": true           // 是否使用 CUDA-aware 通信
    }
  },
  "zero_optimization": { ... }      // 可结合 ZeRO 使用
}
```

**参数**：
- `freeze_step`：在热身期间使用正常的 32 位 Adam，之后切换到 1-bit 压缩。
- `cuda_aware`：如果 NCCL 支持 CUDA-aware，则直接压缩 GPU 张量，减少 CPU 拷贝。

---

## 5. 激活检查点（Activation Checkpointing）

在配置中启用传统激活检查点，减少激活值显存。

```json
{
  "activation_checkpointing": {
    "partition_activations": false,   // 是否开启分区激活检查点（与 ZeRO-3 配合）
    "cpu_checkpointing": false,       // 将检查点卸载到 CPU 内存
    "contiguous_memory_optimization": false,
    "number_checkpoints": null        // 保留的检查点数量（全部或指定整数）
  }
}
```

**分区激活检查点**（Stage 3 推荐）：设置 `"partition_activations": true`，将检查点沿 batch 维度分散到各 GPU。

在代码中需要手动标记哪些模块需要进行检查点重算：

```python
from deepspeed.runtime.activation_checkpointing import checkpointing

# 对指定层开启检查点
checkpointing(model.layers, module_name="transformer_layers")
```

---

## 6. 3D 并行的配置（结合 Megatron-LM）

DeepSpeed 本身不直接实现张量并行（TP）和流水线并行（PP），但能与 Megatron-LM 无缝结合。使用 Megatron-DeepSpeed 启动脚本时，可以在配置文件中指定模型并行参数：

```json
{
  "train_micro_batch_size_per_gpu": 2,
  "tensor_parallel": { "tp_size": 4 },        // 张量并行度
  "pipeline_parallel": { "pp_size": 2 },       // 流水线并行度（需与模型结构匹配）
  "zero_optimization": { "stage": 1 }          // DP 维度上使用 ZeRO-1
}
```

**实际代码中**，这些参数会被 Megatron 框架读取并构建模型。如果使用纯 DeepSpeed 训练自定义模型，TP 和 PP 需要自己在模型定义中实现切分，然后调用 `deepspeed.initialize`。

**推理时的 TP**：DeepSpeed Inference 支持自动张量并行，无需 Megatron（见后文）。

---

## 7. MoE（混合专家）配置

DeepSpeed 内置 MoE 支持，将部分 Transformer FFN 层替换为专家层。

```json
{
  "moe": {
    "enabled": true,
    "ep_size": 4,                    // 专家并行度（专家分布到 ep_size 个 GPU）
    "moe_experts": 64,               // 总专家数
    "top_k": 2,                      // 每个 token 激活的专家数
    "min_capacity": 0,               // 最小容量因子
    "use_residual": false,           // 是否使用残差 MoE
    "expert_parallel_communication_type": "all_to_all", // 通信类型
    "enable_expert_tensor_parallelism": false,   // 是否对单个专家使用 TP
    "moe_token_dropping": true,      // 容量溢出时丢弃 token
    "load_balance": true             // 是否启用负载均衡辅助损失
  }
}
```

**关键参数**：
- `ep_size`：专家并行度，专家会分布在这部分 GPU 上。例如总共 8 张卡，`ep_size=4`，则每张卡持有 16 个专家。
- `moe_experts`：全局专家总数。
- `top_k`：门控选择的专家数量。
- `load_balance`：添加负载均衡损失，防止专家使用不均。

**代码集成**：MoE 层通常使用 DeepSpeed 提供的 `DeepSpeedMoE` 模块替换原有 FFN，或直接使用 Hugging Face 等集成的 MoE 模型。

---

## 8. 推理加速：DeepSpeed-Inference

DeepSpeed 提供了专门的推理引擎，支持内核融合、张量并行和量化。

### 8.1 基本推理用法

```python
import deepspeed
import torch

# 加载原始模型（例如 HuggingFace 模型）
model = AutoModelForCausalLM.from_pretrained("gpt2").half().cuda()

# 初始化推理引擎
ds_engine = deepspeed.init_inference(
    model,
    mp_size=2,                       # 张量并行度
    dtype=torch.float16,
    replace_with_kernel_inject=True, # 注入融合 kernel
    moe_experts=0,                   # 非 MoE 模型设为 0
    checkpoint=None                  # 可传入 DeepSpeed 检查点路径
)

# 推理
output = ds_engine(input_ids)
```

**参数**：
- `mp_size`：模型并行度（张量并行），会自动将模型切分到多张 GPU。
- `replace_with_kernel_inject`：将模型中的算子替换为 DeepSpeed 的高效融合 kernel。
- `dtype`：推理精度（推荐 float16 或 int8）。
- `moe_experts`：若为 MoE 模型，设为专家数（否则 0）。
- `checkpoint`：如果传入 DeepSpeed 检查点路径，会加载分片权重。

### 8.2 使用配置文件进行推理

也可以使用 JSON 配置和 `deepspeed.init_inference` 的 `config` 参数：

```json
{
  "tensor_parallel": {"tp_size": 2},
  "fp16": {"enabled": true},
  "replace_with_kernel_inject": true,
  "kernel_inject": true,
  "enable_cuda_graph": true,         // 开启 CUDA Graph 加速（仅解码阶段）
  "max_tokens": 1024
}
```

**CUDA Graph** 可以捕获重复的计算图，减少 launch overhead，但要求固定输入形状。

---

## 9. Autotuning（自动调优）

DeepSpeed 可以自动搜索最优配置，通过 `deepspeed.autotuning` 运行。

```bash
deepspeed --autotuning run --num_gpus 8 train.py --deepspeed_config ds_config.json
```

或在代码中使用 API：

```python
from deepspeed.autotuning import Autotuner

autotuner = Autotuner(
    model=model,
    train_dataset=dataset,
    args=training_args,
    num_gpus=8,
    max_train_batch_size=256,
    min_train_batch_size=16,
    fp16=True
)
autotuner.run()
```

Autotuner 会尝试不同的 `train_batch_size`、`gradient_accumulation_steps`、ZeRO 阶段和卸载组合，测量吞吐量和显存占用，输出推荐配置。

---

## 10. 保存与加载模型

### 10.1 保存检查点

```python
model_engine.save_checkpoint(save_dir="./checkpoints", tag="step_1000")
```

会生成以下文件：
- `latest`：指向最新 tag 的文件
- `zero_to_fp32.py`：一个脚本，用于将分片权重合并为完整 FP32 模型
- 各 rank 的优化器状态和参数分片（`.pt` 文件）

### 10.2 加载检查点继续训练

```python
model_engine.load_checkpoint(load_dir="./checkpoints", tag="step_1000")
```

### 10.3 导出为 Hugging Face 模型

对于 ZeRO Stage 3，训练时参数被分片，需要先收集：

```bash
python zero_to_fp32.py ./checkpoints/ ./hf_export --tag step_1000
```

或在代码中使用工具：

```python
from deepspeed.utils.zero_to_fp32 import get_fp32_state_dict_from_zero_checkpoint

state_dict = get_fp32_state_dict_from_zero_checkpoint(checkpoint_dir)
model.load_state_dict(state_dict)
```

---

## 11. 使用 Hugging Face Accelerate 集成 DeepSpeed

如果不想直接接触 DeepSpeed API，Accelerate 提供了更简洁的入口。

```python
from accelerate import Accelerator

accelerator = Accelerator(deepspeed_plugin=DeepSpeedPlugin(config_file="ds_config.json"))
model, optimizer, dataloader = accelerator.prepare(model, optimizer, dataloader)

for batch in dataloader:
    outputs = model(**batch)
    loss = outputs.loss
    accelerator.backward(loss)
    optimizer.step()
    optimizer.zero_grad()
```

启动命令：

```bash
accelerate launch --use_deepspeed --deepspeed_config_file ds_config.json train.py
```

Accelerate 会自动将配置传递给 DeepSpeed，无需手动调用 `deepspeed.initialize`。

---

## 12. 常见问题与调优建议

- **OOM**：降低 `train_micro_batch_size_per_gpu`，增加 `gradient_accumulation_steps`，提升 ZeRO stage，开启 CPU/NVMe 卸载，开启激活检查点。
- **速度慢**：增大 `allgather_bucket_size` 和 `reduce_bucket_size`，开启 `overlap_comm`，检查通信拓扑是否合理。
- **单机多卡训练**：直接使用 `deepspeed --num_gpus 8 train.py`。
- **多机多卡**：编写 hostfile，并在每台机器上运行相同命令，指定 `--master_addr` 和 `--master_port`。

---

以上涵盖了 DeepSpeed 从训练到推理的常用 API 和配置参数，结合前面讨论的理论，你可以直接将这些代码片段嵌入自己的项目，并根据模型规模和硬件环境调整配置。如果需要某个更具体场景的示例（例如 RLHF 或 MoE 微调），可以继续深入。

在 Jupyter Notebook 中运行 DeepSpeed 确实会遇到一些特殊情况。因为 DeepSpeed 原生是为命令行分布式启动器（`deepspeed` 或 `torchrun`）设计的，它依赖于启动器注入的环境变量（如 `RANK`, `WORLD_SIZE` 等）。如果在 Jupyter 的 Cell 里直接运行初始化代码，通常会报错或导致 Kernel 崩溃。

业界在 Jupyter/Colab 中跑 DeepSpeed 的**最标准且最稳定**的做法是：**使用 Jupyter 的 `%%writefile` 魔法命令将代码写入 Python 脚本，然后在一个单独的 Cell 中使用 `!deepspeed` 命令去启动它。**


---

### 一、 本例核心知识点总括与实现原理

这个大满贯示例将展示如何在一个极其受限的单卡环境（或多卡）下，通过极限压榨显存来跑通一个复杂模型。

1. **ZeRO-3 (全参数分片) + Offload (CPU 卸载)**
* **实现**：纯配置驱动。在 `ds_config.json` 中配置 `stage: 3`，并将 `offload_optimizer` 和 `offload_param` 指向 `cpu`。DeepSpeed 引擎会自动接管，在用到参数时切入 GPU，用完立刻扔回 CPU 内存。


2. **1-bit Adam (通信压缩)**
* **实现**：纯配置驱动。在 `optimizer` 配置中指定 `OneBitAdam`。它会在前期像普通 Adam 一样训练，达到 `freeze_step` 后，将跨卡通信的梯度压缩为 1-bit，极大节省带宽。


3. **MoE (混合专家架构)**
* **实现**：代码修改。使用 `deepspeed.moe.layer.MoE` 替换标准 Transformer 里的 FFN 层。通过 `ep_size` 控制专家并行度，通过 `gate_loss` 实现专家负载均衡。


4. **Activation Checkpointing (激活值重算)**
* **实现**：代码修改。在模型 `forward` 中，使用 `deepspeed.checkpointing.checkpoint` 包装高显存消耗的计算块。反向传播时 DeepSpeed 会重新计算这部分的激活值，而不是全程保存在显存里。


5. **Inference API (极速推理)**
* **实现**：单独的推理脚本。使用 `deepspeed.init_inference` 接管原生 PyTorch 模型，自动处理分布式切分和算子替换。


---


#### 生成 DeepSpeed 配置文件

这个 Cell 会在当前目录生成 `ds_config.json`。

In [1]:
!mkdir -p ./deepspeed

In [2]:
%%writefile ./deepspeed/ds_config.json
{
  "train_batch_size": 16,
  "gradient_accumulation_steps": 4,
  "train_micro_batch_size_per_gpu": 4,
  "fp16": {
    "enabled": false
  },
  "zero_optimization": {
    "stage": 0
  },
  "activation_checkpointing": {
    "partition_activations": true,
    "cpu_checkpointing": true,
    "contiguous_memory_optimization": false,
    "number_checkpoints": null,
    "synchronize_checkpoint_boundary": false,
    "profile": false
  }
}

Overwriting ./deepspeed/ds_config.json


In [3]:
%%writefile ./deepspeed/train.py
import torch
import argparse
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import deepspeed
from deepspeed.runtime.activation_checkpointing import checkpointing
from deepspeed.moe.layer import MoE

# ==========================================
# 知识点 3 & 4: 定义包含 MoE 和 激活检查点 的模型
# ==========================================
class SimpleMoETransformer(nn.Module):
    def __init__(self, hidden_size=256, num_experts=4, ep_size=1):
        super().__init__()
        # 基础的 Embedding 层
        self.embedding = nn.Linear(128, hidden_size)
        
        # 定义一个简单的 FFN 作为"专家"网络
        expert = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * 4),
            nn.GELU(),
            nn.Linear(hidden_size * 4, hidden_size)
        )
        
        # 知识点 3 (MoE): 使用 DeepSpeed 的 MoE 层替换普通 FFN
        # ep_size: 专家并行度。如果你只有 1 张卡，设为 1；如果有两张卡且设为 2，专家会被平分到两张卡上
        # k=2: 代表 Top-2 路由，即每个 token 会被发送给打分最高的两个专家
        self.moe_layer = MoE(
            hidden_size=hidden_size,
            expert=expert,
            num_experts=num_experts,
            ep_size=ep_size, 
            use_residual=False,
            k=2
        )
        
        self.classifier = nn.Linear(hidden_size, 10)

    def forward(self, x):
        x = self.embedding(x)
        
        # 知识点 4 (Activation Checkpointing): 插入激活检查点
        # 作用：告诉框架“不要保存 moe_layer 内部的中间激活值”，等反向传播到这里时再算一遍，从而省下大笔显存。
        # 注意：checkpoint 函数的输入只能是位置参数 (positional arguments)，所以要包装一个 custom_forward
        def custom_forward(*inputs):
            # MoE 层的输出格式为: (处理后的张量, 门控平衡损失, 专家激活计数)
            moe_out, gate_loss, _ = self.moe_layer(inputs[0])
            return moe_out, gate_loss
            
        # 执行重算包装器
        x, gate_loss = checkpointing.checkpoint(custom_forward, x)
        
        logits = self.classifier(x)
        # 前向传播必须把 gate_loss 传出去，稍后加到总 Loss 里
        return logits, gate_loss

# ==========================================
# 准备伪数据 (不需要关心 batch_size，DeepSpeed 会接管)
# ==========================================
def get_dataloader():
    X = torch.randn(128, 128)
    y = torch.randint(0, 10, (128,))
    dataset = TensorDataset(X, y)
    return DataLoader(dataset)

import json
import torch
import torch.nn as nn
# ... 其他保持一致

def main():
    # ==========================================
    # 增加：标准的命令行参数解析，显式接住 local_rank
    # ==========================================
    parser = argparse.ArgumentParser(description='DeepSpeed Training')
    # DeepSpeed 启动器会自动传入 --local_rank
    parser.add_index = True 
    parser.add_argument('--local_rank', type=int, default=-1,
                        help='local rank passed from distributed runner')
    # ====== 务必添加以下参数，供 DeepSpeed 调优器注入 =====
    parser.add_counts = argparse.ArgumentParser()
    parser.add_argument('--deepspeed_config', type=str, default=None, help="DeepSpeed config path/string")
    parser.add_argument('--per_device_train_batch_size', type=int, default=1, help="Batch size per device")
    parser.add_argument('--gradient_accumulation_steps', type=int, default=1, help="Grad accum steps")
    # ===================================================
    # 这一步非常关键：让 deepspeed 能够通过 args 区分开真正的命令行配置
    args = parser.parse_args()

    with open("./deepspeed/ds_config.json", "r", encoding="utf-8") as f:
            ds_config_dict = json.load(f)

    model = SimpleMoETransformer()
    dataloader = get_dataloader()

    # 🎯 新增：显式创建 PyTorch 原生的 AdamW 优化器
    # 这样 DeepSpeed 就会直接用这个现成的，不再去触发 CPUAdamBuilder() 的本地编译了
    custom_optimizer = torch.optim.AdamW(
        model.parameters(), 
        lr=1e-4, 
        betas=(0.9, 0.999), 
        eps=1e-8, 
        weight_decay=0.01
    )

    # ==========================================
    # 知识点 1 & 2: DeepSpeed 引擎初始化
    # ==========================================
    model_engine, optimizer, trainloader, _ = deepspeed.initialize(
        args=args,
        model=model,
        optimizer=custom_optimizer, # 🎯 核心修改：把我们建好的原生优化器传进去
        training_data=dataloader.dataset,
        config=ds_config_dict
    )
    criterion = nn.CrossEntropyLoss()
    
    # 开始训练循环
    for epoch in range(3):
        model_engine.train()
        for step, batch in enumerate(trainloader):
            # 将数据移动到 model_engine 所在的设备 (GPU)
            inputs = batch[0].to(model_engine.device)
            labels = batch[1].to(model_engine.device)
            
            # 前向传播 (直接调用 engine)
            logits, gate_loss = model_engine(inputs)
            
            # 主任务 Loss
            task_loss = criterion(logits, labels)
            # 核心：MoE 模型必须加上 gate_loss (负载均衡损失)，否则所有 token 都会涌向同一个"聪明"专家
            loss = task_loss + gate_loss
            
            # 反向传播、梯度累积、参数更新 全部交由 engine 接管
            # model_engine.backward 会自动处理 FP16 的 Loss Scaling
            model_engine.backward(loss)
            model_engine.step()
            
            if step % 2 == 0 and model_engine.local_rank == 0:
                print(f"Epoch: {epoch}, Step: {step}, Total Loss: {loss.item():.4f} (Gate Loss: {gate_loss.item():.4f})")

    # ==========================================
    # 保存 ZeRO 模型
    # ==========================================
    # 在 ZeRO-3 下，参数是被分片保存在多张卡上的。
    # save_checkpoint 会保存一个目录，里面包含各个 Rank 的分片文件。
    if model_engine.local_rank == 0:
        print("Saving ZeRO-3 checkpoint...")
    model_engine.save_checkpoint(save_dir="./deepspeed/ds_checkpoint", tag="epoch_2")

if __name__ == "__main__":
    main()

Overwriting ./deepspeed/train.py



---

#### 在 Notebook 中触发训练

使用 `!` 叹号调用终端命令。这里 `--num_gpus 1` 表示使用单卡（如果有两张卡可以改为 2）。DeepSpeed 启动器会自动配置所需的全部环境变量。

In [1]:
import torch
print(torch.cuda.is_available())

True


In [5]:
!deepspeed --num_gpus 1 ./deepspeed/train.py

[2026-06-03 01:23:17,198] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)
 [WARNING]  async_io requires the dev libaio .so object and headers but these were not found.
 [WARNING]  async_io: please install the libaio-dev package with apt
 [WARNING]  If libaio is already installed (perhaps from source), try setting the CFLAGS and LDFLAGS environment variables to where it can be found.
/home/zhangjinrui/anaconda3/envs/flow_planner/lib/python3.9/site-packages/torch/utils/cpp_extension.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging  # type: ignore[attr-defined]
[2026-06-03 01:23:20,594] [WARNING] [runner.py:212:fetch_hostfile] Unable to find hostfile, will proceed with training with local resources only.
[2026

In [6]:
%%writefile ./deepspeed/infer.py
import os
import torch
import deepspeed
from train import SimpleMoETransformer 

def main():
    # ========================================================
    # 核心修复：使用 DeepSpeed 专属的分布式初始化，强行激活 cdb 后端
    # ========================================================
    # 兼容 deepspeed 启动器传进来的 --local_rank 参数
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    torch.cuda.set_device(local_rank)
    
    # 关键：用 deepspeed 自己的 init 代替 torch.distributed.init
    deepspeed.init_distributed(dist_backend="nccl")

    # 1. 实例化纯 FP32 的原生 PyTorch 模型并推到当前 GPU
    model = SimpleMoETransformer().eval().cuda()

    # 2. 用标准 config 字典配置推理引擎
    inference_config = {
        "tensor_parallel": {"tp_size": 1},  
        "dtype": "float",                   # 纯单精度 FP32
        "replace_with_kernel_inject": False,
        "moe": {"moe_experts": [4]}         
    }

    # 此时 cdb 已经完美激活，dist.get_world_size() 顺利通过
    ds_engine = deepspeed.init_inference(
        model=model,
        config=inference_config
    )

    if local_rank == 0:
        print("✅ [DeepSpeed] 推理引擎 (FP32 + MoE 后端) 完美拉起！")

    # 3. 构造严格匹配的 FP32 假数据测试前向
    try:
        dummy_input = torch.randint(0, 1000, (2, 128)).cuda()
        with torch.no_grad():
            res = ds_engine(dummy_input)
    except (RuntimeError, TypeError):
        dummy_input = torch.randn(2, 128).float().cuda()
        with torch.no_grad():
            res = ds_engine(dummy_input)
            
    # 4. 解析输出
    output = res[0] if isinstance(res, (tuple, list)) else res
        
    if local_rank == 0:
        print(f"🚀 [DeepSpeed] 推理成功！输出 Shape: {output.shape}")

if __name__ == "__main__":
    main()

Overwriting ./deepspeed/infer.py



---

#### 运行推理脚本

对于单卡推理，不需要 deepspeed 启动器，直接用原生 python 跑即可。

In [7]:
!deepspeed --num_gpus 1 ./deepspeed/infer.py

[2026-06-03 01:23:41,551] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)
 [WARNING]  async_io requires the dev libaio .so object and headers but these were not found.
 [WARNING]  async_io: please install the libaio-dev package with apt
 [WARNING]  If libaio is already installed (perhaps from source), try setting the CFLAGS and LDFLAGS environment variables to where it can be found.
/home/zhangjinrui/anaconda3/envs/flow_planner/lib/python3.9/site-packages/torch/utils/cpp_extension.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging  # type: ignore[attr-defined]
[2026-06-03 01:23:45,220] [WARNING] [runner.py:212:fetch_hostfile] Unable to find hostfile, will proceed with training with local resources only.
[2026

4. 自动调优 (Autotuning) 演示命令  
如果你不知道上面 ds_config.json 里的 batch size 和 ZeRO 参数怎么设能达到最快速度，可以用下面的命令，让它自己在机器上跑几轮测试：

```bash
deepspeed --autotuning run \
    --num_gpus 1 \
    ./deepspeed/train.py \
    --deepspeed_config ./deepspeed/ds_config.json

```

In [8]:
%%writefile ./deepspeed/ds_config.json
{
  "gradient_accumulation_steps": 4,
  "train_micro_batch_size_per_gpu": 4,
  "fp16": {
    "enabled": false
  },
  "zero_optimization": {
    "stage": 0
  },
  "activation_checkpointing": {
    "partition_activations": true,
    "cpu_checkpointing": true,
    "contiguous_memory_optimization": false,
    "number_checkpoints": null,
    "synchronize_checkpoint_boundary": false,
    "profile": false
  },
  "autotuning": {
    "enabled": true,
    "max_trials": 10,
    "metrics": ["throughput"],
    "zero_stages": [0, 1, 2]
  }
}

Overwriting ./deepspeed/ds_config.json


In [9]:
%%writefile ./deepspeed/train.py
import torch
import argparse
import json
import base64
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import deepspeed
from deepspeed.runtime.activation_checkpointing import checkpointing
from deepspeed.moe.layer import MoE

# ==========================================
# 知识点 3 & 4: 定义包含 MoE 和 激活检查点 的模型
# ==========================================
class SimpleMoETransformer(nn.Module):
    def __init__(self, hidden_size=256, num_experts=4, ep_size=1):
        super().__init__()
        # 基础的 Embedding 层
        self.embedding = nn.Linear(128, hidden_size)
        
        # 定义一个简单的 FFN 作为"专家"网络
        expert = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * 4),
            nn.GELU(),
            nn.Linear(hidden_size * 4, hidden_size)
        )
        
        # 知识点 3 (MoE): 使用 DeepSpeed 的 MoE 层替换普通 FFN
        # ep_size: 专家并行度。如果你只有 1 张卡，设为 1；如果有两张卡且设为 2，专家会被平分到两张卡上
        # k=2: 代表 Top-2 路由，即每个 token 会被发送给打分最高的两个专家
        self.moe_layer = MoE(
            hidden_size=hidden_size,
            expert=expert,
            num_experts=num_experts,
            ep_size=ep_size, 
            use_residual=False,
            k=2
        )
        
        self.classifier = nn.Linear(hidden_size, 10)

    def forward(self, x):
        x = self.embedding(x)
        
        # 知识点 4 (Activation Checkpointing): 插入激活检查点
        # 作用：告诉框架“不要保存 moe_layer 内部的中间激活值”，等反向传播到这里时再算一遍，从而省下大笔显存。
        # 注意：checkpoint 函数的输入只能是位置参数 (positional arguments)，所以要包装一个 custom_forward
        def custom_forward(*inputs):
            # MoE 层的输出格式为: (处理后的张量, 门控平衡损失, 专家激活计数)
            moe_out, gate_loss, _ = self.moe_layer(inputs[0])
            return moe_out, gate_loss
            
        # 执行重算包装器
        x, gate_loss = checkpointing.checkpoint(custom_forward, x)
        
        logits = self.classifier(x)
        # 前向传播必须把 gate_loss 传出去，稍后加到总 Loss 里
        return logits, gate_loss

# ==========================================
# 准备伪数据 (不需要关心 batch_size，DeepSpeed 会接管)
# ==========================================
def get_dataloader():
    X = torch.randn(128, 128)
    y = torch.randint(0, 10, (128,))
    dataset = TensorDataset(X, y)
    return DataLoader(dataset)


def main():
    # ==========================================
    # 增加：标准的命令行参数解析，显式接住 local_rank
    # ==========================================
    parser = argparse.ArgumentParser(description='DeepSpeed Training')
    # DeepSpeed 启动器会自动传入 --local_rank
    parser.add_index = True 
    parser.add_argument('--local_rank', type=int, default=-1,
                        help='local rank passed from distributed runner')
    # ====== 务必添加以下参数，供 DeepSpeed 调优器注入 =====
    parser.add_counts = argparse.ArgumentParser()
    parser.add_argument('--deepspeed_config', type=str, default=None, help="DeepSpeed config path/string")
    parser.add_argument('--per_device_train_batch_size', type=int, default=1, help="Batch size per device")
    parser.add_argument('--gradient_accumulation_steps', type=int, default=1, help="Grad accum steps")
    # ===================================================
    # 这一步非常关键：让 deepspeed 能够通过 args 区分开真正的命令行配置
    args = parser.parse_args()

    with open("./deepspeed/ds_config.json", "r", encoding="utf-8") as f:
            ds_config_dict = json.load(f)

    model = SimpleMoETransformer()
    dataloader = get_dataloader()

    # 🎯 新增：显式创建 PyTorch 原生的 AdamW 优化器
    # 这样 DeepSpeed 就会直接用这个现成的，不再去触发 CPUAdamBuilder() 的本地编译了
    custom_optimizer = torch.optim.AdamW(
        model.parameters(), 
        lr=1e-4, 
        betas=(0.9, 0.999), 
        eps=1e-8, 
        weight_decay=0.01
    )

    # ========================================================
    # 终极物理外挂：拦截 Autotuner 强制注入的 Stage 3
    # ========================================================
    if hasattr(args, 'deepspeed_config') and args.deepspeed_config:
        try:
            # 删除了内部的 import json, base64
            is_base64 = args.deepspeed_config.startswith('eyJ') or args.deepspeed_config.endswith('=')
            
            if is_base64:
                ds_dict = json.loads(base64.b64decode(args.deepspeed_config).decode('utf-8'))
            else:
                with open(args.deepspeed_config, 'r') as f:
                    ds_dict = json.load(f)
            
            if ds_dict.get("zero_optimization", {}).get("stage", 0) == 3:
                ds_dict["zero_optimization"]["stage"] = 0
                print("\n" + "="*60)
                print("🛡️ [安全守卫] 发现 Autotuner 尝试注入 Stage 3！")
                print("🛡️ [安全守卫] 考虑到 MoE 兼容性，已强行将其降级为 Stage 0！")
                print("="*60 + "\n")
                
                if is_base64:
                    args.deepspeed_config = base64.b64encode(json.dumps(ds_dict).encode('utf-8')).decode('utf-8')
        except Exception as e:
            print(f"⚠️ [安全守卫] 解析配置时发生异常，忽略拦截: {e}")
    # ========================================================


    # ==========================================
    # 知识点 1 & 2: DeepSpeed 引擎初始化
    # ==========================================
    model_engine, optimizer, _, _ = deepspeed.initialize(
        args=args,  # ✅ 只要保留 args，deepspeed 会自动从命令行参数中提取动态配置
        model=model,
        config=None  # ✅ 设为 None，避免调优时的参数双重冲突
    )
    
    criterion = nn.CrossEntropyLoss()
    
    # 开始训练循环
    for epoch in range(3):
        model_engine.train()
        for step, batch in enumerate(dataloader):
            # 将数据移动到 model_engine 所在的设备 (GPU)
            inputs = batch[0].to(model_engine.device)
            labels = batch[1].to(model_engine.device)
            
            # 前向传播 (直接调用 engine)
            logits, gate_loss = model_engine(inputs)
            
            # 主任务 Loss
            task_loss = criterion(logits, labels)
            # 核心：MoE 模型必须加上 gate_loss (负载均衡损失)，否则所有 token 都会涌向同一个"聪明"专家
            loss = task_loss + gate_loss
            
            # 反向传播、梯度累积、参数更新 全部交由 engine 接管
            # model_engine.backward 会自动处理 FP16 的 Loss Scaling
            model_engine.backward(loss)
            model_engine.step()
            
            if step % 2 == 0 and model_engine.local_rank == 0:
                print(f"Epoch: {epoch}, Step: {step}, Total Loss: {loss.item():.4f} (Gate Loss: {gate_loss.item():.4f})")

    # ==========================================
    # 保存 ZeRO 模型
    # ==========================================
    # 在 ZeRO-3 下，参数是被分片保存在多张卡上的。
    # save_checkpoint 会保存一个目录，里面包含各个 Rank 的分片文件。
    if model_engine.local_rank == 0:
        print("Saving ZeRO-3 checkpoint...")
    model_engine.save_checkpoint(save_dir="./deepspeed/ds_checkpoint", tag="epoch_2")

if __name__ == "__main__":
    main()

Overwriting ./deepspeed/train.py


In [10]:
!deepspeed --autotuning run \
    --num_gpus 1 \
    ./deepspeed/train.py \
    --deepspeed_config ./deepspeed/ds_config.json

[2026-06-03 01:24:04,808] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)
 [WARNING]  async_io requires the dev libaio .so object and headers but these were not found.
 [WARNING]  async_io: please install the libaio-dev package with apt
 [WARNING]  If libaio is already installed (perhaps from source), try setting the CFLAGS and LDFLAGS environment variables to where it can be found.
/home/zhangjinrui/anaconda3/envs/flow_planner/lib/python3.9/site-packages/torch/utils/cpp_extension.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging  # type: ignore[attr-defined]
[2026-06-03 01:24:08,755] [WARNING] [runner.py:212:fetch_hostfile] Unable to find hostfile, will proceed with training with local resources only.
[2026